# AMEX 30k Challenger — 재현 가능한 성능 도전
실제 성능은 실행 후 확인합니다. 1등 또는 개선을 보장하지 않습니다.

**입력:** 기존 Drive `amex_project/data/processed`의 이력, 레이블, 고정 고객 분할 3개 Parquet.
**설계:** 기존 train+valid 24,000명으로 5-fold OOF 모델 선택. 기존 holdout 6,000명은 마지막 비교에만 사용.
기존 18,000/6,000 검증 수치와 이번 OOF 점수는 직접 비교하지 않습니다. 최신 기록 모델도 같은 CV로 재학습합니다.

**후보:** 최신 기록 LightGBM / 이력 LightGBM 31 leaves / 이력 LightGBM 63 leaves / 이력 CatBoost.
상위 두 모델의 제한된 확률 평균 조합만 비교합니다. 가중치는 OOF에서 선택합니다.
CPU에서 수 시간 이상 걸릴 수 있습니다. 완료한 fold는 자동 저장·재사용합니다. GPU는 필요 없습니다.

**읽는 순서:** 위에서 아래로 실행. 마지막 셀까지 실행하면 holdout이 공개되므로 이후 해당 holdout으로 추가 튜닝하지 마세요.
모델 선택에 사용한 OOF는 낙관적일 수 있습니다. 각 fold의 early stopping도 그 fold를 참조합니다.
이 노트북은 로컬 모델 검증용이며 Kaggle test 예측/제출 파일을 만들지 않습니다.

In [1]:
%pip install -q lightgbm catboost pyarrow scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.3 MB/s eta 0:00:00


In [2]:
from pathlib import Path
import json, time, gc, hashlib, importlib.metadata
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from google.colab import drive

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/amex_project')
DATA = PROJECT / 'data' / 'processed'
SEED = 42
N_FOLDS = 5
THREADS = 2
CAT = ['B_30','B_38','D_114','D_116','D_117','D_120','D_126','D_63','D_64','D_66','D_68']
history = pd.read_parquet(DATA / 'customer_history_30k.parquet')
labels = pd.read_parquet(DATA / 'customer_labels_30k.parquet').set_index('customer_ID')
split = pd.read_parquet(DATA / 'customer_split_30k.parquet').set_index('customer_ID')
assert labels.index.is_unique and split.index.is_unique
assert set(history.customer_ID) == set(labels.index) == set(split.index)
assert set(split['split']) == {'train','valid','holdout'}
assert not history.duplicated(['customer_ID','S_2']).any()
assert labels.target.isin([0,1]).all()
history['S_2'] = pd.to_datetime(history['S_2'])
assert history.S_2.notna().all()
dev_ids = split.index[split['split'].isin(['train','valid'])].sort_values()
hold_ids = split.index[split['split'].eq('holdout')].sort_values()
y = labels.loc[dev_ids, 'target'].to_numpy()
print('Development:', len(dev_ids), 'Holdout:', len(hold_ids))

Mounted at /content/drive
Development: 24000 Holdout: 6000


## 1. AMEX metric
**정상(0) 가중치 20, 부도(1) 가중치 1**을 사용합니다.
AMEX = (weighted normalized Gini + weighted top-4% default capture) / 2.
상위 4%는 원시 고객 수 4%가 아니라 **누적 가중치** 기준입니다. 학습 손실에는 자동으로 20배를 적용하지 않습니다.
동점은 입력 고객 ID 순서로 안정적으로 처리합니다.
출처: https://www.kaggle.com/competitions/amex-default-prediction/overview/evaluation

In [3]:
def amex_parts(y_true, score):
    y_true = np.asarray(y_true, dtype=np.float64)
    score = np.asarray(score, dtype=np.float64)
    assert y_true.shape == score.shape and np.isfinite(score).all()
    assert 0 < y_true.sum() < len(y_true)
    def gini(order):
        yy = y_true[order]
        ww = np.where(yy == 0, 20.0, 1.0)
        population = np.cumsum(ww) / ww.sum()
        positives = np.cumsum(yy * ww) / np.sum(yy * ww)
        return np.sum((positives - population) * ww)
    order = np.argsort(-score, kind='stable')
    yy = y_true[order]
    ww = np.where(yy == 0, 20.0, 1.0)
    selected = np.cumsum(ww) <= int(0.04 * ww.sum())
    capture = yy[selected].sum() / y_true.sum()
    normalized_gini = gini(order) / gini(np.argsort(-y_true, kind='stable'))
    return {'amex': (normalized_gini + capture) / 2,
            'gini': normalized_gini, 'capture4': capture,
            'auc': roc_auc_score(y_true, score)}

def lgb_amex(y_true, score):
    return 'amex', amex_parts(y_true, score)['amex'], True

# 완벽한 순위의 점검 (부도 비중 25%에서 4% 가중 예산으로 모든 부도 포착 가능)
test_y = np.tile([0,0,0,1], 100)
assert np.isclose(amex_parts(test_y, test_y)['amex'], 1.0)

## 2. 고객 이력 변수
최신값 + mean/std/min/max/유효 관측수, 최초값 대비 차이, 직전 관측 대비 차이,
최근 3회 평균과 전체 평균 차이, 결측비율, 관측일수, 범주 고유값 수·변경 횟수.
최근 3회는 정확한 3개월과 다릅니다. 결측에서/으로의 변화는 범주 변경 횟수에서 제외합니다.
고객별 원자료만 사용하므로 고객 간 학습 통계 누출은 없습니다. 범주 사전은 각 학습 fold에서만 만듭니다.

In [4]:
def build_features(raw):
    h = raw.sort_values(['customer_ID','S_2']).copy()
    numeric = [c for c in h.columns if c not in CAT + ['customer_ID','S_2']]
    g = h.groupby('customer_ID', sort=True)
    last = h.drop_duplicates('customer_ID', keep='last').set_index('customer_ID')
    first = h.drop_duplicates('customer_ID', keep='first').set_index('customer_ID')
    latest = last.drop(columns='S_2').sort_index()
    stats = g[numeric].agg(['mean','std','min','max','count'])
    stats.columns = [f'{a}__{b}' for a,b in stats.columns]
    mean = g[numeric].mean()
    delta_first = (last[numeric] - first[numeric]).add_suffix('__delta_first')
    previous = g[numeric].shift(1)
    delta_previous = h[numeric] - previous
    delta_previous['customer_ID'] = h.customer_ID
    delta_previous = delta_previous.drop_duplicates('customer_ID', keep='last').set_index('customer_ID').add_suffix('__delta_previous')
    recent = g.tail(3).groupby('customer_ID')[numeric].mean()
    recent_delta = (recent - mean).add_suffix('__recent3_minus_mean')
    last_delta = (last[numeric] - mean).add_suffix('__last_minus_mean')
    counts = g.size().rename('n_statements')
    missing = (1 - g[numeric].count().div(counts, axis=0)).add_suffix('__missing_rate')
    meta = pd.DataFrame({'n_statements': counts,
        'history_days': (g.S_2.max()-g.S_2.min()).dt.days})
    cat_unique = g[CAT].nunique().add_suffix('__nunique')
    prior_cat = g[CAT].shift(1)
    changed = (h[CAT].ne(prior_cat) & h[CAT].notna() & prior_cat.notna()).fillna(False)
    changed['customer_ID'] = h.customer_ID
    changes = changed.groupby('customer_ID').sum().add_suffix('__changes')
    rich = latest.join([stats, delta_first, delta_previous, recent_delta,
                        last_delta, missing, meta, cat_unique, changes])
    for frame in (latest, rich):
        num = [c for c in frame.columns if c not in CAT]
        frame[num] = frame[num].replace([np.inf,-np.inf], np.nan).astype('float32')
        for col in CAT:
            frame[col] = frame[col].astype('string')
        assert frame.index.is_unique and frame.columns.is_unique
    return latest, rich

latest, rich = build_features(history)
views = {'latest': latest, 'rich': rich}
print('Latest:', latest.shape, 'History:', rich.shape)
print('Feature memory MB:', sum(f.memory_usage(deep=True).sum() for f in views.values()) / 1024**2)

Latest: (30000, 188) History: (30000, 1982)
Feature memory MB: 285.08706855773926


## 3. 5-fold 후보 비교와 자동 재개
각 후보의 미학습 고객 예측(OOF)을 저장합니다. LightGBM은 AMEX로 early stopping.
CatBoost는 AUC로 early stopping한 뒤 검증 예측을 50-tree 간격으로 평가해 AMEX 최적 지점을 선택합니다.
기존 최신 기록 모델 설정을 같은 CV로 재학습하여 이력 추가와 모델 변경 효과를 비교합니다.
데이터·설정·함수·패키지 버전 해시별로 결과 폴더가 구분됩니다.

In [5]:
CONFIGS = {
 'latest_lgb': dict(view='latest', kind='lgb', leaves=31, child=100, l2=1.0, rounds=1500),
 'history_lgb31': dict(view='rich', kind='lgb', leaves=31, child=100, l2=5.0, rounds=2500),
 'history_lgb63': dict(view='rich', kind='lgb', leaves=63, child=60, l2=10.0, rounds=2500),
 'history_cat6': dict(view='rich', kind='cat', depth=6, rounds=2500),
}
versions = {p: importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','lightgbm','catboost']}

def prepare(train, valid, kind):
    a, b = train.copy(), valid.copy()
    for col in CAT:
        if kind == 'lgb':
            dtype = pd.CategoricalDtype(categories=a[col].dropna().unique().tolist())
            a[col], b[col] = a[col].astype(dtype), b[col].astype(dtype)
        else:
            a[col] = a[col].fillna('__MISSING__').astype(str)
            b[col] = b[col].fillna('__MISSING__').astype(str)
    return a, b

def make_model(cfg, seed, rounds=None):
    rounds = cfg['rounds'] if rounds is None else rounds
    if cfg['kind'] == 'lgb':
        return lgb.LGBMClassifier(objective='binary', metric='None',
            n_estimators=rounds, learning_rate=.03, num_leaves=cfg['leaves'],
            min_child_samples=cfg['child'], colsample_bytree=.8,
            reg_lambda=cfg['l2'], random_state=seed, n_jobs=THREADS,
            deterministic=True, force_col_wise=True, verbosity=-1)
    return CatBoostClassifier(iterations=rounds, depth=cfg['depth'],
        learning_rate=.04, loss_function='Logloss', eval_metric='AUC',
        l2_leaf_reg=8, random_seed=seed, thread_count=THREADS,
        allow_writing_files=False, verbose=False)

# Code/config/data fingerprint prevents accidental reuse after edits.
import marshal
fingerprint = hashlib.sha256()
for frame in (history, labels.sort_index(), split.sort_index()):
    fingerprint.update(pd.util.hash_pandas_object(frame, index=True).values.tobytes())
fingerprint.update(json.dumps([CONFIGS, versions, SEED, N_FOLDS, THREADS], sort_keys=True).encode())
for fn in (amex_parts, build_features, prepare, make_model):
    fingerprint.update(marshal.dumps(fn.__code__))
RUN = PROJECT / 'experiments' / ('challenger_' + fingerprint.hexdigest()[:12])
RUN.mkdir(parents=True, exist_ok=True)
(RUN / 'environment.json').write_text(json.dumps(versions, indent=2))
print('Outputs:', RUN)

folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(dev_ids, y))
oof, best_rounds, records = {}, {}, []
for name, cfg in CONFIGS.items():
    oof[name] = np.zeros(len(dev_ids))
    best_rounds[name] = []
    for fold, (it, iv) in enumerate(folds):
        dest = RUN / f'{name}_fold{fold}.joblib'
        if dest.exists():
            result = joblib.load(dest)
            assert np.array_equal(result['valid_ids'], dev_ids[iv].to_numpy())
        else:
            start = time.perf_counter()
            a,b = prepare(views[cfg['view']].loc[dev_ids[it]],
                          views[cfg['view']].loc[dev_ids[iv]], cfg['kind'])
            m = make_model(cfg, SEED + fold)
            if cfg['kind'] == 'lgb':
                m.fit(a, y[it], categorical_feature=CAT,
                      eval_set=[(b,y[iv])], eval_metric=lgb_amex,
                      callbacks=[lgb.early_stopping(150, first_metric_only=True, verbose=False)])
                rounds = m.best_iteration_
                pred = m.predict_proba(b)[:,1]
            else:
                m.fit(a, y[it], cat_features=CAT, eval_set=(b,y[iv]),
                      early_stopping_rounds=150, use_best_model=True, verbose=False)
                checkpoints = sorted(set(list(range(50,m.tree_count_+1,50)) + [m.tree_count_]))
                trials = [(r,m.predict_proba(b,ntree_end=r)[:,1]) for r in checkpoints]
                rounds,pred = max(trials, key=lambda z: amex_parts(y[iv],z[1])['amex'])
                m.shrink(rounds)
            result = dict(valid_ids=dev_ids[iv].to_numpy(), pred=pred,
                          rounds=int(rounds), seconds=time.perf_counter()-start)
            joblib.dump(dict(model=m, columns=a.columns.tolist(),
                             categories={c:a[c].cat.categories.tolist() for c in CAT} if cfg['kind']=='lgb' else None),
                        RUN / f'{name}_fold{fold}_model.joblib')
            joblib.dump(result,dest)
            del a,b,m
            gc.collect()
        oof[name][iv] = result['pred']
        best_rounds[name].append(result['rounds'])
        row = dict(model=name, fold=fold, rounds=result['rounds'],
                   **amex_parts(y[iv],result['pred']))
        records.append(row)
        print(row, flush=True)
    pd.DataFrame(records).to_csv(RUN / 'fold_metrics.csv',index=False)

scores = pd.DataFrame([dict(model=n, **amex_parts(y,p)) for n,p in oof.items()]).sort_values('amex',ascending=False)
display(scores)
oof_frame = pd.DataFrame(oof,index=dev_ids)
oof_frame.index.name='customer_ID'
oof_frame.to_parquet(RUN / 'oof_predictions.parquet')
scores.to_csv(RUN / 'oof_metrics.csv',index=False)

Outputs: /content/drive/MyDrive/amex_project/experiments/challenger_33482abad5e1
{'model': 'latest_lgb', 'fold': 0, 'rounds': 426, 'amex': np.float64(0.7842858947446608), 'gini': np.float64(0.9185315642278574), 'capture4': np.float64(0.6500402252614642), 'auc': np.float64(0.9592764745436406)}
{'model': 'latest_lgb', 'fold': 1, 'rounds': 143, 'amex': np.float64(0.7598057826145355), 'gini': np.float64(0.9122101171196582), 'capture4': np.float64(0.6074014481094127), 'auc': np.float64(0.9561165806560031)}
{'model': 'latest_lgb', 'fold': 2, 'rounds': 312, 'amex': np.float64(0.7713876353741266), 'gini': np.float64(0.9112386657603208), 'capture4': np.float64(0.6315366049879324), 'auc': np.float64(0.9556309824757183)}
{'model': 'latest_lgb', 'fold': 3, 'rounds': 197, 'amex': np.float64(0.7406484343708042), 'gini': np.float64(0.9012485984278512), 'capture4': np.float64(0.580048270313757), 'auc': np.float64(0.950637259968729)}
{'model': 'latest_lgb', 'fold': 4, 'rounds': 290, 'amex': np.float64(

,model,amex,gini,capture4,auc
3,history_cat6,0.771511,0.915669,0.627353,0.957837
2,history_lgb63,0.768474,0.914904,0.622043,0.957454
1,history_lgb31,0.765700,0.914184,0.617216,0.957094
0,latest_lgb,0.760696,0.909968,0.611424,0.954986


## 4. OOF로 앙상블 선택 · 설정 확정
개별 상위 두 모델에 대해 가중치 0/0.25/0.5/0.75/1만 비교합니다.
단일 모델이 더 좋으면 그대로 선택합니다. 후보 수를 늘리면 검증 과적합 위험도 증가합니다.

In [6]:
first, second = scores.model.iloc[:2]
blend_trials=[]
for weight in [0.,.25,.5,.75,1.]:
    p = weight*oof[first] + (1-weight)*oof[second]
    blend_trials.append(dict(weight_first=weight, **amex_parts(y,p)))
blend_table=pd.DataFrame(blend_trials).sort_values('amex',ascending=False)
display(blend_table)
w=float(blend_table.iloc[0].weight_first)
weights={n:float(v) for n,v in [(first,w),(second,1-w)] if v>0}
locked={'weights':weights,
        'rounds':{n:int(np.median(best_rounds[n])) for n in CONFIGS},
        'configs':CONFIGS, 'seed':SEED, 'development_customers':len(dev_ids),
        'holdout_customers':len(hold_ids)}
(RUN/'locked_selection.json').write_text(json.dumps(locked,indent=2))
print('Selected:',weights)
print('Prediction correlation:', np.corrcoef(oof[first],oof[second])[0,1])

,weight_first,amex,gini,capture4,auc
3,0.75,0.771668,0.916786,0.626549,0.958395
4,1.00,0.771511,0.915669,0.627353,0.957837
1,0.25,0.770416,0.916536,0.624296,0.958270
2,0.50,0.768926,0.917095,0.620756,0.958550
0,0.00,0.768474,0.914904,0.622043,0.957454


Selected: {'history_cat6': 0.75, 'history_lgb63': 0.25}
Prediction correlation: 0.9871300473656942


## 5. 최종 평가 — 여기서부터 holdout 공개
선택된 모델과 최신 기록 비교 모델을 동일한 개발 고객 24,000명으로 재학습합니다.
반복 횟수는 CV 최적 횟수 중앙값으로 고정. holdout을 early stopping/가중치 선택에 사용하지 않습니다.
아래 셀은 실행 시 최종 점수를 공개합니다. 결과 확인 후 이 holdout으로 재튜닝하면 더 이상 최종 평가가 아닙니다.
기존 18,000명 기준 모델과 비교하는 대신 학습 고객 수를 맞추어 공정하게 비교합니다.

In [7]:
final_path=RUN/'holdout_predictions.parquet'
if final_path.exists():
    final=pd.read_parquet(final_path)
    print('이미 공개한 최종 평가 결과를 재사용합니다.')
else:
    final=pd.DataFrame(index=hold_ids)
    for name in set(weights) | {'latest_lgb'}:
        cfg=CONFIGS[name]
        a,b=prepare(views[cfg['view']].loc[dev_ids],views[cfg['view']].loc[hold_ids],cfg['kind'])
        m=make_model(cfg,SEED,locked['rounds'][name])
        if cfg['kind']=='lgb':
            m.fit(a,y,categorical_feature=CAT)
        else:
            m.fit(a,y,cat_features=CAT,verbose=False)
        final[name]=m.predict_proba(b)[:,1]
        joblib.dump(dict(model=m,columns=a.columns.tolist(),
                         categories={c:a[c].cat.categories.tolist() for c in CAT} if cfg['kind']=='lgb' else None),
                    RUN/f'{name}_final.joblib')
        del a,b,m
        gc.collect()
    final['selected']=sum(v*final[n] for n,v in weights.items())
    final['target']=labels.loc[hold_ids,'target']
    final.index.name='customer_ID'
    final.to_parquet(final_path)

final_scores=pd.DataFrame([
    dict(model=n,**amex_parts(final.target,final[n]))
    for n in ['latest_lgb','selected']])
display(final_scores)
final_scores.to_csv(RUN/'final_metrics.csv',index=False)

# Paired stratified bootstrap: same sampled customers for both models.
# Fixed fitted models only; does not include retraining/split uncertainty.
rng=np.random.default_rng(SEED)
yt=final.target.to_numpy()
p0=final.latest_lgb.to_numpy(); p1=final.selected.to_numpy()
i0=np.flatnonzero(yt==0); i1=np.flatnonzero(yt==1)
diffs=[]
for _ in range(300):
    idx=np.r_[rng.choice(i0,len(i0),replace=True),rng.choice(i1,len(i1),replace=True)]
    rng.shuffle(idx)
    diffs.append(amex_parts(yt[idx],p1[idx])['amex']-amex_parts(yt[idx],p0[idx])['amex'])
ci=np.quantile(diffs,[.025,.975])
delta=amex_parts(yt,p1)['amex']-amex_parts(yt,p0)['amex']
print(f'Holdout AMEX difference: {delta:+.5f}; approximate 95% interval [{ci[0]:+.5f}, {ci[1]:+.5f}]')
print('구간이 0을 포함하면 개선을 단정하지 마세요. 실제 대회 순위는 이 결과로 알 수 없습니다.')
(RUN/'bootstrap_summary.json').write_text(json.dumps(dict(delta=delta,low=float(ci[0]),high=float(ci[1]))))

,model,amex,gini,capture4,auc
0,latest_lgb,0.750303,0.898546,0.602061,0.949284
1,selected,0.770858,0.906171,0.635544,0.953095


Holdout AMEX difference: +0.02055; approximate 95% interval [+0.00922, +0.03401]
구간이 0을 포함하면 개선을 단정하지 마세요. 실제 대회 순위는 이 결과로 알 수 없습니다.


89

# 제출

In [8]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/amex_project")

RUN = (
    PROJECT
    / "experiments"
    / "challenger_33482abad5e1"
)

locked = json.loads(
    (RUN / "locked_selection.json").read_text()
)

submit_weights = locked["weights"]
submit_configs = locked["configs"]

submit_models = {
    name: joblib.load(RUN / f"{name}_final.joblib")
    for name in submit_weights
}

assert np.isclose(sum(submit_weights.values()), 1.0)
assert all(weight >= 0 for weight in submit_weights.values())

print("제출 모델:", submit_weights)

제출 모델: {'history_cat6': 0.75, 'history_lgb63': 0.25}


In [10]:
%pip install -q kaggle

In [11]:
import os
from getpass import getpass

# GitHub 토큰이 아니라 Kaggle 토큰입니다.
if not os.environ.get("KAGGLE_API_TOKEN"):
    os.environ["KAGGLE_API_TOKEN"] = getpass(
        "Kaggle API 토큰 입력: "
    )

Kaggle API 토큰 입력: ··········


In [12]:
from pathlib import Path
import shutil
import subprocess
import pandas as pd

RAW = Path("/content/amex/raw")
RAW.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(RAW).free / 1024**3
print(f"Colab 남은 디스크 공간: {free_gb:.1f} GB")


def ensure_local_file(filename):
    candidates = [
        RAW / filename,
        RAW / f"{filename}.zip",
    ]

    # 이미 다운로드한 파일이 있으면 재사용
    for path in candidates:
        if path.is_file() and path.stat().st_size > 0:
            print(f"로컬 파일 사용: {path.name}")
            return path

    print(f"Kaggle에서 다운로드: {filename}", flush=True)

    subprocess.run(
        [
            "kaggle", "competitions", "download",
            "amex-default-prediction",
            "-f", filename,
            "-p", str(RAW),
        ],
        check=True,
    )

    for path in candidates:
        if path.is_file() and path.stat().st_size > 0:
            print(
                f"다운로드 완료: {path.name} "
                f"({path.stat().st_size / 1024**3:.2f} GB)"
            )
            return path

    raise FileNotFoundError(
        f"다운로드 후에도 파일을 찾을 수 없습니다: {filename}"
    )


# 작은 파일부터 받아 인증과 다운로드 확인
template_path = ensure_local_file("sample_submission.csv")
test_path = ensure_local_file("test_data.csv")

template = pd.read_csv(
    template_path,
    dtype={"customer_ID": "string"},
)

assert template["customer_ID"].is_unique
assert list(template.columns) == ["customer_ID", "prediction"]

print(f"제출 대상 고객: {len(template):,}명")

Colab 남은 디스크 공간: 86.7 GB
Kaggle에서 다운로드: sample_submission.csv
다운로드 완료: sample_submission.csv.zip (0.03 GB)
Kaggle에서 다운로드: test_data.csv
다운로드 완료: test_data.csv.zip (13.76 GB)
제출 대상 고객: 924,621명


In [13]:
def predict_ensemble_block(block):
    assert block["S_2"].notna().all()
    assert not block.duplicated(
        ["customer_ID", "S_2"]
    ).any()

    # Challenger 노트북에서 정의한 함수
    latest_block, rich_block = build_features(block)

    feature_views = {
        "latest": latest_block,
        "rich": rich_block,
    }

    ids = latest_block.index

    blended = np.zeros(len(ids), dtype=np.float64)

    for name, weight in submit_weights.items():
        config = submit_configs[name]
        bundle = submit_models[name]

        X = (
            feature_views[config["view"]]
            .loc[ids, bundle["columns"]]
            .copy()
        )

        if config["kind"] == "lgb":
            # 최종 모델 학습 때 저장한 범주 사전 사용
            for col in CAT:
                cat_dtype = pd.CategoricalDtype(
                    categories=bundle["categories"][col]
                )

                X[col] = X[col].astype(cat_dtype)

        elif config["kind"] == "cat":
            for col in CAT:
                X[col] = (
                    X[col]
                    .astype("string")
                    .fillna("__MISSING__")
                    .astype(str)
                )

        else:
            raise ValueError(f"지원하지 않는 모델: {config['kind']}")

        pred = bundle["model"].predict_proba(X)[:, 1]

        assert np.isfinite(pred).all()
        assert ((pred >= 0) & (pred <= 1)).all()

        blended += weight * pred

    return pd.DataFrame({
        "customer_ID": ids.to_numpy(),
        "prediction": blended,
    })

In [14]:
import time

# 원본 테스트 열 구조 확인
test_columns = pd.read_csv(
    test_path,
    nrows=0,
).columns.tolist()

assert set(["customer_ID", "S_2"] + CAT).issubset(test_columns)

numeric_columns = [
    col for col in test_columns
    if col not in ["customer_ID", "S_2"] + CAT
]

dtype_map = {
    col: "float32"
    for col in numeric_columns
}

dtype_map.update({
    col: "string"
    for col in CAT + ["customer_ID"]
})

prediction_parts = []
carry = None
last_seen_id = None

rows_read = 0
customers_predicted = 0
started = time.perf_counter()

with pd.read_csv(
    test_path,
    dtype=dtype_map,
    parse_dates=["S_2"],
    chunksize=50_000,
) as reader:

    for chunk_no, chunk in enumerate(reader, start=1):
        chunk_ids = chunk["customer_ID"]

        assert chunk_ids.notna().all()
        assert chunk_ids.is_monotonic_increasing, (
            "원본의 고객 ID 정렬을 확인해야 합니다."
        )

        if last_seen_id is not None:
            assert chunk_ids.iloc[0] >= last_seen_id, (
                "청크 간 고객 ID 순서가 맞지 않습니다."
            )

        last_seen_id = chunk_ids.iloc[-1]
        rows_read += len(chunk)

        if carry is not None:
            chunk = pd.concat(
                [carry, chunk],
                ignore_index=True,
            )

        last_customer = chunk["customer_ID"].iloc[-1]
        is_last = chunk["customer_ID"].eq(last_customer)

        carry = chunk.loc[is_last].copy()
        complete = chunk.loc[~is_last]

        if not complete.empty:
            result = predict_ensemble_block(complete)
            prediction_parts.append(result)
            customers_predicted += len(result)

        if chunk_no == 1 or chunk_no % 10 == 0:
            minutes = (time.perf_counter() - started) / 60

            print(
                f"읽은 행 {rows_read:,} | "
                f"예측 고객 {customers_predicted:,} | "
                f"{minutes:.1f}분",
                flush=True,
            )

# 마지막 고객까지 처리
if carry is not None and not carry.empty:
    prediction_parts.append(
        predict_ensemble_block(carry)
    )

ensemble_predictions = pd.concat(
    prediction_parts,
    ignore_index=True,
)

assert ensemble_predictions["customer_ID"].is_unique

print(
    "전체 예측 완료:",
    f"{len(ensemble_predictions):,}명"
)

읽은 행 50,000 | 예측 고객 4,086 | 0.1분
읽은 행 500,000 | 예측 고객 40,694 | 1.2분
읽은 행 1,000,000 | 예측 고객 81,357 | 2.3분
읽은 행 1,500,000 | 예측 고객 122,011 | 3.4분
읽은 행 2,000,000 | 예측 고객 162,616 | 4.5분
읽은 행 2,500,000 | 예측 고객 203,364 | 5.7분
읽은 행 3,000,000 | 예측 고객 244,056 | 6.8분
읽은 행 3,500,000 | 예측 고객 284,777 | 8.1분
읽은 행 4,000,000 | 예측 고객 325,446 | 9.3분
읽은 행 4,500,000 | 예측 고객 366,175 | 10.4분
읽은 행 5,000,000 | 예측 고객 406,810 | 11.6분
읽은 행 5,500,000 | 예측 고객 447,542 | 12.8분
읽은 행 6,000,000 | 예측 고객 488,255 | 14.0분
읽은 행 6,500,000 | 예측 고객 528,889 | 15.1분
읽은 행 7,000,000 | 예측 고객 569,549 | 16.2분
읽은 행 7,500,000 | 예측 고객 610,199 | 17.4분
읽은 행 8,000,000 | 예측 고객 650,897 | 18.6분
읽은 행 8,500,000 | 예측 고객 691,579 | 19.8분
읽은 행 9,000,000 | 예측 고객 732,209 | 21.0분
읽은 행 9,500,000 | 예측 고객 772,853 | 22.2분
읽은 행 10,000,000 | 예측 고객 813,534 | 23.3분
읽은 행 10,500,000 | 예측 고객 854,295 | 24.5분
읽은 행 11,000,000 | 예측 고객 895,030 | 25.6분
전체 예측 완료: 924,621명


In [15]:
assert set(ensemble_predictions["customer_ID"]) == set(
    template["customer_ID"]
), "테스트 고객이 누락됐거나 추가됐습니다."

submission = template[["customer_ID"]].merge(
    ensemble_predictions,
    on="customer_ID",
    how="left",
    sort=False,
    validate="one_to_one",
)

assert list(submission.columns) == list(template.columns)
assert len(submission) == len(template)

assert np.array_equal(
    submission["customer_ID"].to_numpy(),
    template["customer_ID"].to_numpy(),
)

assert np.isfinite(submission["prediction"]).all()
assert submission["prediction"].between(0, 1).all()

SUBMISSION_DIR = PROJECT / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_PATH = (
    SUBMISSION_DIR
    / "challenger_cat75_lgb25_submission.csv"
)

submission.to_csv(
    SUBMISSION_PATH,
    index=False,
    float_format="%.9f",
)

print("저장 완료:", SUBMISSION_PATH)
display(submission.head())

저장 완료: /content/drive/MyDrive/amex_project/submissions/challenger_cat75_lgb25_submission.csv


,customer_ID,prediction
0,00000469ba478561f23a92a868bd366de6f6527a684c9a...,0.030606
1,00001bf2e77ff879fab36aa4fac689b9ba411dae63ae39...,0.002120
2,0000210045da4f81e5f122c6bde5c2a617d03eef67f82c...,0.024035
3,00003b41e58ede33b8daf61ab56d9952f17c9ad1c3976c...,0.204197
4,00004b22eaeeeb0ec976890c1d9bfc14fd9427e98c4ee9...,0.859736


In [16]:
import subprocess

subprocess.run(
    [
        "kaggle", "competitions", "submit",
        "amex-default-prediction",
        "-f", str(SUBMISSION_PATH),
        "-m", "History CatBoost 75% + LightGBM 25%; 24k training customers",
    ],
    check=True,
)

CompletedProcess(args=['kaggle', 'competitions', 'submit', 'amex-default-prediction', '-f', '/content/drive/MyDrive/amex_project/submissions/challenger_cat75_lgb25_submission.csv', '-m', 'History CatBoost 75% + LightGBM 25%; 24k training customers'], returncode=0)

In [17]:
!kaggle competitions submissions amex-default-prediction

fileName                               date                        description                                                  status                     publicScore  privateScore  
-------------------------------------  --------------------------  -----------------------------------------------------------  -------------------------  -----------  ------------  
challenger_cat75_lgb25_submission.csv  2026-09-16 14:22:49.913000  History CatBoost 75% + LightGBM 25%; 24k training customers  SubmissionStatus.PENDING                              
transformer_v1_submission.csv          2026-09-16 13:02:55.070000  Transformer v1 - trained on 18k customers                    SubmissionStatus.COMPLETE  0.76408      0.77455       
